# Classification NBA Model

## Configuration

## Imports

In [38]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training
)
from sklearn.model_selection import cross_validate, train_test_split
from xgboost import XGBClassifier



## Load Data

In [39]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260310.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)


In [3]:
df_stats

,TOTAL_LINE_bet365,TEAM_ID_TEAM_HOME,TEAM_CITY_TEAM_HOME,TEAM_ABBREVIATION_TEAM_HOME,TEAM_NAME_TEAM_HOME,MATCHUP_TEAM_HOME,GAME_NUMBER_TEAM_HOME,TEAM_ID_TEAM_AWAY,TEAM_CITY_TEAM_AWAY,TEAM_ABBREVIATION_TEAM_AWAY,...,TRAVEL_RECENCY_RATIO_AWAY_2D_OVER_14D_BEFORE,REST_DAYS_DIFF_HOME_MINUS_AWAY_BEFORE,INJURY_PTS_SHARE_HOME_BEFORE,INJURY_PTS_SHARE_AWAY_BEFORE,STAR_PTS_PCT_DIFF_HOME_MINUS_AWAY_BEFORE,POSS_X_TSPCT_HOME_BEFORE,POSS_X_TSPCT_AWAY_BEFORE,IS_WEEKEND_BEFORE,MONTH_BEFORE,IS_US_HOLIDAY_BEFORE
0,NaN,1610612739,Cleveland,CLE,Cleveland Cavaliers,CLE vs. BOS,1,1610612738,Boston,BOS,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
1,NaN,1610612744,Golden State,GSW,Golden State Warriors,GSW vs. HOU,1,1610612745,Houston,HOU,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
2,NaN,1610612765,Detroit,DET,Detroit Pistons,DET vs. CHA,1,1610612766,Charlotte,CHA,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
3,NaN,1610612754,Indiana,IND,Indiana Pacers,IND vs. BKN,1,1610612751,Brooklyn,BKN,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
4,NaN,1610612753,Orlando,ORL,Orlando Magic,ORL vs. MIA,1,1610612748,Miami,MIA,...,NaN,0,NaN,NaN,NaN,NaN,NaN,0,10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11180,226.0,1610612739,Cleveland,CLE,Cleveland Cavaliers,CLE vs. PHI,65,1610612755,Philadelphia,PHI,...,0.849136,-1,0.188118,0.764287,0.105331,58.997566,53.184225,0,3,0
11181,220.0,1610612751,Brooklyn,BKN,Brooklyn Nets,BKN vs. MEM,64,1610612763,Memphis,MEM,...,0.000000,0,0.299512,0.991088,-0.004616,55.295096,57.772822,0,3,0
11182,237.5,1610612760,Oklahoma City,OKC,Oklahoma City Thunder,OKC vs. DEN,66,1610612743,Denver,DEN,...,0.000000,-1,0.449673,0.234545,0.028966,59.064355,61.419909,0,3,0
11183,226.0,1610612762,Utah,UTA,Utah Jazz,UTA vs. GSW,65,1610612744,Golden State,GSW,...,0.706994,0,0.901535,0.854227,0.070674,60.097087,57.559721,0,3,0


In [4]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=50, max_na_per_row=3, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11185 rows
Basic cleaning complete: 8576 rows remaining

Starting advanced column cleaning with 3097 columns

Advanced column cleaning complete: 3097 → 2086 columns (1011 removed)


Dropping NA rows for SEASON_YEAR 2017...
   Removed 0 rows with NaN values from 2017 season

Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 0 (0.0%)
  Critical columns requiring data: 4
  Columns zero-filled: 98
  Infer pairs applied: 20/136
  Remaining NaN cells: 944087

Dropping rows with more than 3 NaN values...
Removed 3967 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (4609, 2086)


In [5]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1895,total_consensus_pct_under,62,1.35,2024.0
1896,spread_consensus_pct_home,56,1.22,2024.0
1902,ml_consensus_opener_price_away,14,0.30,2025.0
1903,ml_consensus_opener_price_home,14,0.30,2025.0
1348,DIFF_FROM_LINE_draftkings_LAST_ALL_1_MATCHES_D...,4,0.09,2022.0
1749,total_consensus_pct_over_TREND_SLOPE_LAST_5_HO...,3,0.07,2023.0
769,DIFF_FROM_LINE_draftkings_LAST_ALL_1_MATCHES_B...,3,0.07,2022.0
1751,total_consensus_pct_under_TREND_SLOPE_LAST_5_H...,2,0.04,2023.0
1755,spread_consensus_pct_home_TREND_SLOPE_LAST_5_H...,2,0.04,2023.0
1912,total_draftkings_price_over,2,0.04,2022.0


In [6]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [7]:
print(f" Total Seasons in dataset: {df_to_train['SEASON_YEAR'].nunique()}")
print(f" Total Games in dataset: {len(df_to_train)}")
print(f"Seasons: {df_to_train['SEASON_YEAR'].unique()}")
#print number of games per season
print(df_to_train['SEASON_YEAR'].value_counts())

 Total Seasons in dataset: 5
 Total Games in dataset: 4609
Seasons: [2021 2022 2023 2024 2025]
SEASON_YEAR
2024    1248
2022    1171
2021    1157
2025     882
2023     151
Name: count, dtype: int64


## Train / Test

In [23]:
X = df_to_train.drop(["TOTAL_POINTS", "SEASON_YEAR", "GAME_DATE"], axis=1, errors="ignore")
X = df_to_train.drop(["TOTAL_POINTS",  "GAME_DATE"], axis=1, errors="ignore")

y = pd.to_numeric(df_to_train["TOTAL_POINTS"], errors="coerce")


In [24]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.075, random_state=16, shuffle=False)

In [25]:
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
# Check number of coulmns
print(f"Number of columns in training set: {X_train.shape[1]}")
print(f"Number of columns in test set: {X_test.shape[1]}")

Training set size: 4263
Test set size: 346
Number of columns in training set: 2084
Number of columns in test set: 2084


## Cross-validation

In [26]:
from sklearn.model_selection import TimeSeriesSplit, cross_validate, cross_val_predict
from sklearn.metrics import mean_squared_error, mean_absolute_error, make_scorer, root_mean_squared_error


In [27]:
# Declare time-series cross-validation strategy
kf = TimeSeriesSplit(n_splits=5, test_size = 30, gap = 2)


In [28]:
import numpy as np


def over_under_betting_accuracy(
    true_total: np.ndarray,
    pred_total: np.ndarray,
    betting_line: np.ndarray,
) -> float:
    true_total = np.asarray(true_total, dtype=float)
    pred_total = np.asarray(pred_total, dtype=float)
    betting_line = np.asarray(betting_line, dtype=float)

    valid = np.isfinite(true_total) & np.isfinite(pred_total) & np.isfinite(betting_line)
    if not np.any(valid):
        return np.nan

    true_total = true_total[valid]
    pred_total = pred_total[valid]
    betting_line = betting_line[valid]

    true_side = np.where(
        true_total > betting_line,
        "OVER",
        np.where(true_total < betting_line, "UNDER", "PUSH"),
    )
    pred_side = np.where(
        pred_total > betting_line,
        "OVER",
        np.where(pred_total < betting_line, "UNDER", "PUSH"),
    )

    non_push = (true_side != "PUSH") & (pred_side != "PUSH")
    if not np.any(non_push):
        return np.nan

    return float(np.mean(true_side[non_push] == pred_side[non_push]))


class OverUnderScorer:
    """Custom sklearn-compatible scorer for over/under betting accuracy using bet365 line."""

    def __init__(self, line_col: str):
        self.line_col = line_col

    def __call__(self, estimator, X, y_true):
        if self.line_col not in X.columns:
            raise KeyError(f"{self.line_col} not found in X for scoring")

        y_pred = estimator.predict(X)
        betting_line = pd.to_numeric(X[self.line_col], errors="coerce").to_numpy(dtype=float)

        return over_under_betting_accuracy(
            true_total=np.asarray(y_true, dtype=float),
            pred_total=np.asarray(y_pred, dtype=float),
            betting_line=betting_line,
        )


over_under_scorer = OverUnderScorer(BET365_LINE_COL)


In [29]:
# Declare scores to be used
scoring = {
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(root_mean_squared_error),
    'MAE': make_scorer(mean_absolute_error),
    'OU_Betting_Accuracy': over_under_scorer,
}

In [30]:
def print_metrics(cv_results):
    for sc in scoring.keys():
        if sc == 'OU_Betting_Accuracy':
            print(f'Train {sc}:', f"{cv_results[f'train_{sc}'].mean():.2%}")
            print(f'Validation {sc}:', f"{cv_results[f'test_{sc}'].mean():.2%}")
        else:
            print(f'Train {sc}:', cv_results[f'train_{sc}'].mean().round(5))
            print(f'Validation {sc}:', cv_results[f'test_{sc}'].mean().round(5))
        print()

In [31]:
def real_vs_pred(model, X_train, y_train):
    preds = cross_val_predict(model, X_train, y_train, cv=kf, n_jobs=-1)
    x_line = np.arange(y_train.min(), y_train.max())
    plt.scatter(y_train, preds)
    plt.plot(x_line, x_line, color='orange')
    plt.xlabel('Real target')
    plt.ylabel('Predicted target')
    plt.show()

## Baseline

In [32]:
from sklearn.dummy import DummyRegressor

season_bl = DummyRegressor(strategy='mean')
cv_results = cross_validate(season_bl, X_train, y_train, cv=kf,
                            scoring=scoring, return_train_score=True)
season_bl.fit(X_train, y_train)
print_metrics(cv_results)

Train MSE: 383.46609
Validation MSE: 343.20835

Train RMSE: 19.58229
Validation RMSE: 18.43692

Train MAE: 15.63722
Validation MAE: 14.86168

Train OU_Betting_Accuracy: 49.91%
Validation OU_Betting_Accuracy: 53.33%



In [33]:
# Baseline 3: Predict the opening betting line as total points


y_pred_baseline_3 = pd.to_numeric(X_train['TOTAL_LINE_consensus_opener'], errors="coerce")
valid = y_pred_baseline_3.notna() & pd.to_numeric(y_train, errors="coerce").notna()

# Evaluate
mse = mean_squared_error(y_train[valid], y_pred_baseline_3[valid])
mae = mean_absolute_error(y_train[valid], y_pred_baseline_3[valid])
rmse = root_mean_squared_error(y_train[valid], y_pred_baseline_3[valid])
OU_accuracy = over_under_betting_accuracy(
    true_total=y_train[valid],
    pred_total=y_pred_baseline_3[valid],
    betting_line=X_train.loc[valid, 'TOTAL_LINE_bet365']
)
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"Over/Under Accuracy: {OU_accuracy:.2f}")    

MSE: 307.65
RMSE: 17.54
MAE: 13.98
Over/Under Accuracy: 0.49


## Logistic Regression

In [34]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
cv_results = cross_validate(lr, X_train.fillna(0), y_train, cv=kf,
                            scoring=scoring, return_train_score=True)

print_metrics(cv_results)

Train MSE: 186.04727
Validation MSE: 1070.01541

Train RMSE: 13.63988
Validation RMSE: 32.46793

Train MAE: 10.61048
Validation MAE: 25.67473

Train OU_Betting_Accuracy: 70.92%
Validation OU_Betting_Accuracy: 46.00%



In [36]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBRegressor

# Example XGBoost regressor:
xgb_reg = XGBRegressor(
    max_depth=4,
    learning_rate=0.02,
    n_estimators=350,
    subsample=0.5,       # Equivalent to max_samples in GBRegressor
    colsample_bytree=0.6, # Equivalent to max_features in GBRegressor
    n_jobs=-1,
    random_state=16)

cv_results = cross_validate(
    xgb_reg, 
    X_train, y_train, 
    cv=kf, 
    scoring=scoring,      # Use your custom scoring or e.g. 'neg_mean_absolute_error'
    return_train_score=True,
    n_jobs=-1
)

# Print metrics
print_metrics(cv_results)
# Train final model on full train set
xgb_reg.fit(X_train, y_train)

Train MSE: 165.77137
Validation MSE: 286.53158

Train RMSE: 12.87517
Validation RMSE: 16.71571

Train MAE: 10.24162
Validation MAE: 13.74961

Train OU_Betting_Accuracy: 82.70%
Validation OU_Betting_Accuracy: 47.33%



XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.02, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=350, n_jobs=-1,
             num_parallel_tree=None, random_state=16, ...)

In [37]:
feature_importances = xgb_reg.feature_importances_
important_features = np.argsort(feature_importances)[::-1]  
feature_importances = pd.DataFrame({
    'Feature': X_train.columns[important_features],
    'Importance': feature_importances[important_features]
}).sort_values(by="Importance", ascending=False)
feature_importances



,Feature,Importance
0,TOTAL_LINE_draftkings,0.004565
1,TOTAL_LINE_caesars,0.004490
2,TOTAL_LINE_fanduel,0.003953
3,TOTAL_LINE_betmgm,0.003114
4,TOTAL_LINE_bet365,0.001835
...,...,...
1775,ml_consensus_opener_price_LAST_ALL_5_MATCHES_B...,0.000000
1776,ml_draftkings_price_LAST_HOME_AWAY_5_MATCHES_B...,0.000000
1777,spread_consensus_pct_away_SEASON_BEFORE_STD_DI...,0.000000
1778,SAME_DIVISION_BEFORE,0.000000


In [38]:
import numpy as np
import pandas as pd


def evaluate_ou_thresholds(
    model,
    X_test: pd.DataFrame,
    y_test_total: pd.Series,
    line_col: str,
    thresholds=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10),
):
    """Evaluate OU accuracy at different confidence thresholds using bet365 line."""
    if line_col not in X_test.columns:
        raise KeyError(f"{line_col} not found in X_test")

    y_pred_total = np.asarray(model.predict(X_test), dtype=float)
    y_true_total = pd.to_numeric(y_test_total, errors="coerce").to_numpy(dtype=float)
    line = pd.to_numeric(X_test[line_col], errors="coerce").to_numpy(dtype=float)

    margin = np.abs(y_pred_total - line)

    rows = []
    n_total = len(y_true_total)

    for t in thresholds:
        mask = margin > t
        n = int(mask.sum())

        if n == 0:
            acc = np.nan
        else:
            acc = over_under_betting_accuracy(
                true_total=y_true_total[mask],
                pred_total=y_pred_total[mask],
                betting_line=line[mask],
            )

        rows.append(
            {
                "threshold_abs_pred_vs_line_gt": t,
                "n_games": n,
                "pct_of_test": (n / n_total) if n_total else np.nan,
                "ou_accuracy": acc,
            }
        )

    return pd.DataFrame(rows), y_pred_total


results_df, y_pred_test_total = evaluate_ou_thresholds(
    model=xgb_reg,
    X_test=X_test,
    y_test_total=y_test,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_vs_line_gt,n_games,pct_of_test,ou_accuracy
0,0,346,100.0%,51.48%
1,1,256,74.0%,52.80%
2,2,164,47.4%,51.88%
3,3,93,26.9%,53.26%
4,4,44,12.7%,50.00%
5,5,23,6.6%,56.52%
6,6,9,2.6%,55.56%
7,7,3,0.9%,66.67%
8,8,3,0.9%,66.67%
9,9,2,0.6%,50.00%


# Optuna

In [116]:
import numpy as np
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit

# Use forward-chaining folds for time-ordered validation
kf = TimeSeriesSplit(n_splits=5)


def ou_accuracy_with_threshold_totals(
    true_total: np.ndarray,
    pred_total: np.ndarray,
    betting_line: np.ndarray,
    threshold_abs: float = 0.0,
    min_coverage: float = 0.25,
):
    # "Edge" in points relative to the line
    margin = np.abs(pred_total - betting_line)

    # Bet only when predicted edge exceeds threshold
    mask = margin > threshold_abs
    coverage = float(np.mean(mask))

    if coverage < min_coverage or mask.sum() == 0:
        return 0.0, coverage

    score = over_under_betting_accuracy(
        true_total=true_total[mask],
        pred_total=pred_total[mask],
        betting_line=betting_line[mask],
    )

    # If all filtered as pushes/no-valid, penalize
    if np.isnan(score):
        return 0.0, coverage

    return score, coverage


def _predict_best(model, X):
    # Use best iteration if early stopping happened
    if getattr(model, "best_iteration", None) is not None:
        # Newer XGBoost
        try:
            return model.predict(X, iteration_range=(0, model.best_iteration + 1))
        except TypeError:
            # Older compatibility path
            ntree_limit = getattr(model, "best_ntree_limit", None)
            if ntree_limit is not None:
                return model.predict(X, ntree_limit=ntree_limit)
    return model.predict(X)


def objective(trial, X, y):
    if BET365_LINE_COL not in X.columns:
        raise KeyError(f"{BET365_LINE_COL} not found in X")

    threshold_abs = trial.suggest_float("threshold_abs", 0.0, 10.0, step=0.5)
    min_coverage = 0.25

    params = {
        "booster": "gbtree",
        "tree_method": "hist",
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "min_child_weight": trial.suggest_float("min_child_weight", 5.0, 20.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.50, 0.80),

        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.30, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 20.0, log=True),

        "n_estimators": trial.suggest_int("n_estimators", 225, 500, step=25),
        "eval_metric": "rmse",
        "early_stopping_rounds": 200,

        "random_state": 16,
        "n_jobs": -1,
    }

    fold_scores = []
    fold_coverages = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr = y.iloc[tr_idx].to_numpy(dtype=float)
        y_va = y.iloc[va_idx].to_numpy(dtype=float)

        model = XGBRegressor(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        y_pred_total = _predict_best(model, X_va)
        line_va = pd.to_numeric(X_va[BET365_LINE_COL], errors="coerce").to_numpy(dtype=float)

        score, coverage = ou_accuracy_with_threshold_totals(
            true_total=y_va,
            pred_total=np.asarray(y_pred_total, dtype=float),
            betting_line=line_va,
            threshold_abs=threshold_abs,
            min_coverage=min_coverage,
        )

        fold_scores.append(score)
        fold_coverages.append(coverage)

        trial.report(float(np.mean(fold_scores)), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trial.set_user_attr("mean_coverage", float(np.mean(fold_coverages)))
    return float(np.mean(fold_scores))

# ----------------------------
# Run the search (2 to 3 hours)
# ----------------------------
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=16),
    pruner=MedianPruner(n_warmup_steps=2),
)

study.optimize(lambda t: objective(t, X_train, y_train), timeout =3.3 * 3600, n_jobs=1)

print("Best value (CV OU success):", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(k, v)

# ----------------------------
# Train final model on full train set
# ----------------------------
best_params = study.best_params.copy()
best_threshold = best_params.pop("threshold_abs")  # strategy threshold

final_params = {
    "booster": "gbtree",
    "tree_method": "hist",
    "random_state": 16,
    "n_jobs": -1,
    **best_params,
}

final_model = XGBRegressor(**final_params)



[I 2026-03-08 21:08:44,363] A new study created in memory with name: no-name-008f0131-9235-40c6-b408-afef66f572da


[I 2026-03-08 21:10:50,885] Trial 0 finished with value: 0.5367685124581956 and parameters: {'threshold_abs': 2.0, 'max_depth': 4, 'min_child_weight': 10.728161890750807, 'gamma': 0.22800975066403162, 'subsample': 0.6164373012086636, 'colsample_bytree': 0.5669242825073867, 'learning_rate': 0.05082275623350138, 'reg_alpha': 0.004517786465395959, 'reg_lambda': 0.01706650116663653, 'n_estimators': 500}. Best is trial 0 with value: 0.5367685124581956.
[I 2026-03-08 21:12:21,071] Trial 1 finished with value: 0.0 and parameters: {'threshold_abs': 5.5, 'max_depth': 2, 'min_child_weight': 13.615793124046794, 'gamma': 0.7922608676158321, 'subsample': 0.550168784046414, 'colsample_bytree': 0.5880461768306593, 'learning_rate': 0.05316051828678, 'reg_alpha': 0.07195423366266383, 'reg_lambda': 0.05127746977873438, 'n_estimators': 350}. Best is trial 0 with value: 0.5367685124581956.
[I 2026-03-08 21:15:37,074] Trial 2 finished with value: 0.5329787222237634 and parameters: {'threshold_abs': 1.0, 'm

Best value (CV OU success): 0.5636575596116018
Best params:
threshold_abs 2.5
max_depth 2
min_child_weight 13.458385267100654
gamma 1.6787783971720915
subsample 0.5375445218986955
colsample_bytree 0.5322509133896342
learning_rate 0.06670270351832987
reg_alpha 0.018575387631018136
reg_lambda 0.03967617102507207
n_estimators 500


## Check in Test set

In [117]:
final_model.fit(X_train, y_train, verbose=False)
results_df_final, y_pred_test = evaluate_ou_thresholds(
    model=final_model,
    X_test=X_test,
    y_test_total=y_test,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),  # 0..10
)
# Pretty display
display(results_df_final.style.format({
    "pct_of_test": "{:.1%}",
    "error_betting_accuracy": "{:.2%}",
}))


,threshold_abs_pred_vs_line_gt,n_games,pct_of_test,ou_accuracy
0,0,460,100.0%,0.493392
1,1,391,85.0%,0.496104
2,2,322,70.0%,0.474684
3,3,244,53.0%,0.474790
4,4,181,39.3%,0.477273
5,5,133,28.9%,0.484375
6,6,88,19.1%,0.464286
7,7,63,13.7%,0.475410
8,8,46,10.0%,0.488889
9,9,36,7.8%,0.514286


In [118]:
feature_importances = final_model.feature_importances_
important_features = np.argsort(feature_importances)[::-1]  
feature_importances = pd.DataFrame({
    'Feature': X_train.columns[important_features],
    'Importance': feature_importances[important_features]
}).sort_values(by="Importance", ascending=False)
feature_importances

,Feature,Importance
0,TOTAL_LINE_draftkings,0.010004
1,TOTAL_LINE_caesars,0.008176
2,TOTAL_LINE_fanduel,0.007336
3,TOTAL_LINE_betmgm,0.005289
4,TOTAL_LINE_bet365,0.003757
...,...,...
1187,DIFF_FROM_LINE_fanduel_LAST_ALL_2_MATCHES_BEFO...,0.000000
1188,DIFF_FROM_LINE_draftkings_LAST_ALL_1_MATCHES_B...,0.000000
1189,DIFF_FROM_LINE_draftkings_LAST_5_MINUS_LAST_10...,0.000000
1190,DIFF_FROM_LINE_draftkings_LAST_5_MINUS_LAST_10...,0.000000


In [119]:

import json
from pathlib import Path

import joblib
import numpy as np
from xgboost import XGBRegressor


def train_final_model(
    X_all,
    y_all,
    study,
    *,
    use_early_stopping: bool = True,
    val_frac: float = 0.1,
    random_state: int = 16,
):
    """
    Train a final XGBRegressor using Optuna best params.
    Also returns the betting threshold (threshold_abs) stored in the study params.
    """
    best_params = study.best_params.copy()
    best_threshold = float(best_params.pop("threshold_abs"))

    final_params = {
        "booster": "gbtree",
        "tree_method": "hist",
        "random_state": random_state,
        "n_jobs": -1,
        "eval_metric": "rmse",
        **best_params,
    }

    model = XGBRegressor(**final_params)

    if not use_early_stopping:
        # Train on all data without early stopping
        model.fit(X_all, y_all, verbose=False)
        return model, best_threshold

    # Keep early stopping by holding out a small validation split
    n = len(X_all)
    rng = np.random.default_rng(random_state)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_val = max(1, int(round(val_frac * n)))
    val_idx = idx[:n_val]
    tr_idx = idx[n_val:]

    X_tr, y_tr = X_all.iloc[tr_idx], y_all.iloc[tr_idx]
    X_va, y_va = X_all.iloc[val_idx], y_all.iloc[val_idx]

    model.set_params(early_stopping_rounds=200)
    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False,
    )
    return model, best_threshold


def save_model_bundle(
    model: XGBRegressor,
    threshold_abs: float,
    feature_names: list[str],
    out_dir: str | Path,
    name: str = "xgb_total_points_ou_model",
):
    """
    Saves:
      - model.joblib (sklearn wrapper)
      - metadata.json (threshold + feature order)
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    model_path = out_dir / f"{name}.joblib"
    meta_path = out_dir / f"{name}.meta.json"

    joblib.dump(model, model_path)

    metadata = {
        "threshold_abs": float(threshold_abs),
        "feature_names": list(feature_names),
        "betting_line_column": BET365_LINE_COL,
        "target": "TOTAL_POINTS",
        "xgboost_version_note": "Saved via joblib XGBRegressor wrapper",
    }
    meta_path.write_text(json.dumps(metadata, indent=2))

    return model_path, meta_path


def load_model_bundle(model_path: str | Path, meta_path: str | Path):
    model = joblib.load(model_path)
    metadata = json.loads(Path(meta_path).read_text())
    return model, metadata


# -------------------------
# Example usage
# -------------------------

# 1) Build ALL data
X_all = X
y_all = y

# 2) Train final model
final_model, best_threshold = train_final_model(
    X_all=X_all,
    y_all=y_all,
    study=study,
    use_early_stopping=True,
    val_frac=0.1,
    random_state=16,
)

# 3) Save model + metadata (threshold + feature ordering)
model_path, meta_path = save_model_bundle(
    model=final_model,
    threshold_abs=best_threshold,
    feature_names=list(X_all.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/",
    name="drop_na_cols_50_recent_data_xgb_total_points_07_03_26",
)

print("Saved:", model_path)
print("Saved:", meta_path)

# 4) Load later
loaded_model, meta = load_model_bundle(model_path, meta_path)
print("Loaded threshold_abs:", meta["threshold_abs"])


Saved: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/drop_na_cols_50_recent_data_xgb_total_points_07_03_26.joblib
Saved: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/drop_na_cols_50_recent_data_xgb_total_points_07_03_26.meta.json
Loaded threshold_abs: 2.5
